# noise_reachability step 9: do noises that solved one resistant state help on other states of their task? (Kaggle, 2 x T4)

Step 9 of `noise_reachability/README.md` (design fixed 2026-11-03). The base `lerobot/smolvla_libero` at `a24f1f3804`,
`n_action_steps=10`, family `repeat`, tasks 0, 4, 6, 7: a library of earlier hit noises against fresh noises on eight
held-out resistant states. No training and no dataset input.

**Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing. The wrapper check runs first, then
the harness: a failure of either stops the run. Then the scene checks, the library and random episodes, and the analysis.
If anything stops the run, step 13 still writes the zip, then reports the failure as an error.

**One session should do (about 3.4 h, an estimate).** If step 12 says work remains, add this version's output as input
(**Add Input > Your Work**), and only that version, and Save & Run All again: finished episodes are skipped. Step 6 must
then print `RESUMING`.

Download from the Output tab: `step9_results.zip`.

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "SET_AFTER_COMMIT"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 9; nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [0, 4, 6, 7]
HELD_OUT = {0: [14, 27], 4: [18, 26], 6: [14, 20], 7: [38, 47]}   # step 8's eligible states it did not test
N_ACTION_STEPS = 10
OUT_NAME = "step9"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
STOP_AT_H = 10.5                   # no new work after this many hours of the session; running jobs are stopped
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 9 at {REPO_REF}: tasks {TASKS}, held-out states {HELD_OUT}, n_action_steps {N_ACTION_STEPS}, family repeat, "
      f"at most {120 + 542} episodes (120 harness, 142 library, 400 random), output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
previous = [p for p in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True))
            if os.path.isdir(p) and os.path.exists(f"{p}/repo_commit.txt")]
assert len(previous) <= 1, f"attach only the latest version's output, found {previous}"
for prev in previous:
    print("RESUMING from", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
if not previous:
    print("FRESH START: no earlier output attached. If this is the second session, stop and add it as input.", flush=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 13, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






SEARCH = f"cd {REPO} && {PY} -u noise_reachability/deep_noise_search.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}"
SCENE = f"cd {REPO} && {PY} -u noise_reachability/check_scene.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}"
ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step9.py --root {OUT_ROOT}"


def task_dir(task):
    return f"{OUT_ROOT}/task_{task}"


def run_stage(name, jobs, total):
    """True when the stage has no work left: a job that failed stops the run, the deadline leaves work for a next version."""
    print(f"{name}: {len(jobs)} of {total} jobs to run; {hours_left():.1f} h left", flush=True)
    if not jobs:
        return True
    if hours_left() <= 0:
        return False
    codes = run_jobs(jobs, max_parallel=4)
    if failed(codes):
        fail(f"{name} failed: {failed(codes)}")
        return False
    return all(c == 0 for c in codes.values()) and len(codes) == len(jobs)


def run_gate(which):
    if sh(f"{ANALYZE} --gate {which}", check=False, env=RUN_ENV).returncode != 0:
        fail(f"the {which} gate failed: investigate before spending more")
        return False
    return True


def listed(flag):
    """The lines of an analysis listing as lists of words, or None after failing the run."""
    r = sh(f"{ANALYZE} {flag}", check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"the listing {flag} failed")
        return None
    return [line.split() for line in r.stdout.strip().splitlines()]


def scene_checked(task, state):
    """One state's scene check, in its own process, one at a time."""
    if hours_left() <= 0:
        return False
    r = sh(f"{SCENE} --task_id {task} --init_state {state} --output_dir {task_dir(task)}/scene", check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"the scene check failed to run on task {task} state {state}")
        return False
    return True

## 8. Wrapper check on each task's real model (a few minutes); stops the run if it fails

In [ ]:
CHECK_DONE = True
for t in TASKS:
    r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step5a.py --task_id {t} --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}",
           check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"check_step5a failed on task {t}")
        CHECK_DONE = False

## 9. Harness: the base on seeds 0-2, per task; stops the run if it fails

In [ ]:
jobs = {}
for t in TASKS:
    for s in (0, 1, 2):
        out = f"{task_dir(t)}/harness/seed{s}"
        if len(read_csv(f"{out}/per_episode_base.csv")) < 10:
            jobs[f"harness_task{t}_seed{s}"] = (
                f"cd {REPO} && {PY} -u noise_reachability/eval_noise_selection.py --task_id {t} --policies base "
                f"--policy_path {POLICY_PATH} --seeds {s} --init_states 0-9 --n_action_steps {N_ACTION_STEPS} --output_dir {out}",
                len(jobs) % 2)
HARNESS_DONE = CHECK_DONE and run_stage("harness", jobs, 12) and run_gate("harness")

## 10. Scene checks of the eight held-out states

A state that fails is dropped, not replaced: it is written to `scene_state<i>.json`, and the report lists it.

In [ ]:
SCENES_DONE = HARNESS_DONE
if HARNESS_DONE:
    for t in TASKS:
        for s in HELD_OUT[t]:
            if SCENES_DONE and not scene_checked(t, s):
                SCENES_DONE = False

## 11. Library and random noises on every held-out state read

In [ ]:
TESTS_DONE = False
plan = listed("--jobs") if SCENES_DONE else None
if plan is not None:
    jobs = {}
    for t, s, arm, ids in plan:
        out = f"{task_dir(t)}/test/{arm}"
        if len(read_csv(f"{out}/per_episode_repeat_state{s}.csv")) < len(ids.split(",")):
            jobs[f"test_{arm}_task{t}_state{s}"] = (
                f"{SEARCH} --task_id {t} --family repeat --init_state {s} --noise_ids {ids} --output_dir {out}", len(jobs) % 2)
    print(f"{len(plan)} jobs planned: {[(t, s, arm, len(ids.split(','))) for t, s, arm, ids in plan]}", flush=True)
    TESTS_DONE = run_stage("tests", jobs, len(plan))

## 12. Analysis (when every episode is done)

In [ ]:
if TESTS_DONE:
    if sh(f"{ANALYZE} --out {OUT_ROOT}/step9_report.txt", check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")
elif not FAILURE:
    print("WORK REMAINS: add this version's output as input and Save & Run All again")

## 13. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step9_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step9_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)